# GPU device support verification (PR #112)

Verifies `Model.to(device, dtype)` on a real GPU before merging
[PR #112](https://github.com/JBjoernskov/Twin4Build/pull/112).

**Setup**: Runtime > Change runtime type > **T4 GPU**, then Runtime > Run all.

1. Installs Twin4Build from the `feature/gpu-device-support` branch.
2. Runs the device test suite (`test_device.py`): expect **6 passed** --
   4 CPU tests plus the 2 CUDA tests (fp64 simulate parity, fast estimation
   objective built and validated on `cuda`). On a CPU runtime the 2 CUDA
   tests skip.
3. Quick timing/accuracy check: the same simulation on cpu/fp64, cuda/fp64,
   cuda/fp32. Expect cuda to be *slower* at batch size 1 (that is the
   documented expectation) and the max temperature deviation to be ~1e-12 K
   for cuda/fp64 and below ~1e-3 K for cuda/fp32.

**Troubleshooting**: if a cell fails with a numpy `ImportError` after the
install (pip replaced numpy under the live kernel), do Runtime > **Restart
session**, then Run all again -- the install cell is idempotent. To pick up
new commits on the branch, delete the runtime (Runtime > Disconnect and
delete runtime) so the repo is cloned fresh.

**Troubleshooting**: if a cell fails with a numpy ImportError after the
install (pip replaced numpy under the live kernel), do Runtime > **Restart
session**, then Run all again -- the install cell is idempotent. To pick up
new commits on the branch, delete the runtime (Runtime > Disconnect and
delete runtime) so the repo is cloned fresh.


In [ ]:
# Clone (or update) the PR branch and install without touching Colab's
# preinstalled scientific stack (replacing numpy mid-session breaks the kernel).
import importlib.metadata as _md
_pins = " ".join(
    f'"{_p}=={_md.version(_p)}"'
    for _p in ("numpy", "scipy", "pandas", "matplotlib")
)
!git clone --depth 1 -b feature/gpu-device-support https://github.com/JBjoernskov/Twin4Build.git 2>/dev/null || git -C Twin4Build pull
!pip install -q ./Twin4Build {_pins}

import numpy
import torch
print("numpy", numpy.__version__)
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU -- switch the runtime type to T4 GPU for the full check.")


In [ ]:
!cd Twin4Build && python -m pytest twin4build/tests/simulator/test_device.py -v


In [ ]:
# Runs as a subprocess (like the pytest cell) so it works even if the
# notebook kernel's numpy import state is stale.
!cd Twin4Build && python twin4build/examples/gpu_verify_timing.py